# פייז 7 — A, A+PDE, B, C1, C1+PDE

השוואה בין loss פיזיקלי רך, projection קשיח ומבנה פיזיקלי בארכיטקטורה.
**B הוא B-loop**: FNO עם projection לשימור מסה במהלך האימון ובזמן החיזוי. אין עליו סוויפ נוסף.
המחברת בודקת checkpoints קיימים בלבד. אימון הניסויים החסרים נעשה ב־`00_training.ipynb`.


## פתיחה ב־Colab
העלו מחברת זו ל־Colab. בתחילת runtime חדש תא ההכנה יבקש את `spno-colab-source.zip` המצורף, אלא אם הקוד המעודכן כבר נמצא ב־`PROJECT_ROOT`. גם runtime עם חבילת קוד ישנה יבקש את החבילה המעודכנת. חיבור Drive נעשה דרך ממשק Colab הרגיל.

הגדירו את נתיב תוצרי פייז 6. הנתיב המקורי יכול להיות ב־Drive או בדיסק המקומי של הריצה שטרם נסגרה. אפשר להעתיק את התוצרים ל־Drive באמצעות `COPY_TO`; המקור אינו נמחק. חשוב להשלים את ההעתקה לפני סגירת runtime שבו התוצרים נמצאים רק תחת `/content`.

נדרשת תיקיית המקור המלאה, עם checkpoints וקובצי `train.pt`, `val.pt`, `test.pt`, וכן פרוטוקול האימון המקורי. ארכיון Phase 6 המיועד לבדיקה בלבד ומשמיט את train.pt אינו מספיק לפייז 7. השתמשו בחבילת הקוד המעודכנת כדי לקבל גם את C1+PDE.

המחברת קוראת משקולות קיימות. B-loop משתמש במשקולות שאומנו עם projection; B-post אינו חלק מההשוואה הזו. checkpoints ישנים אינם מכילים optimizer ולכן אינם נקודת המשך מדויקת לאימון שנקטע. כל אימון חדש דרך הקטלוג שומר התקדמות בסוף כל epoch.


**איתור תוצרים:** ברירת המחדל `SOURCE_ROOT=None` מחפשת תיקייה מלאה בנתיבי Phase 6 המקומיים וב־MyDrive.
אם נמצאה רק חבילת `phase6-standalone-artifacts-*.zip` מלאה, היא נחלצת לתיקייה מקומית נפרדת.
אפשר גם להגדיר `SOURCE_ROOT` ישירות ל־ZIP או לתיקייה שמכילה את התוצרים בתוכה.
אם נמצאו כמה ריצות מלאות, מוצגים הנתיבים לבחירה מפורשת. קובץ הקוד `spno-colab-source.zip` אינו מכיל משקולות או דאטה.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, importlib.util

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
PROJECT_ROOT = Path(os.environ.get("SPNO_PROJECT_ROOT", "/content/spno-colab" if IN_COLAB else str(Path.cwd())))
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if IN_COLAB:
    from google.colab import drive, files
    drive.mount("/content/drive")
    artifact_code = PROJECT_ROOT / "src/spno/artifacts.py"
    needs_source_update = (not artifact_code.is_file()
                           or "def resolve_standalone_source(" not in artifact_code.read_text())
    if needs_source_update:
        # Upload the updated source bundle, including artifact discovery.
        import zipfile
        uploaded = files.upload()
        archives = [name for name in uploaded if name.endswith(".zip")]
        if len(archives) != 1:
            raise ValueError("Upload the single spno-colab-source.zip bundle")
        PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archives[0]) as archive:
            for member in archive.infolist():
                if not (PROJECT_ROOT / member.filename).resolve().is_relative_to(PROJECT_ROOT.resolve()):
                    raise ValueError("Unsafe archive member")
            archive.extractall(PROJECT_ROOT)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_ROOT) + "[dirichlet,notebooks]"])
if not (PROJECT_ROOT / "src/spno/workflow.py").is_file():
    raise FileNotFoundError("Set PROJECT_ROOT to the updated spno source directory")
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))
OPTIONS = json.loads(os.environ.get("SPNO_OPTIONS", "{}"))
from IPython.display import display, HTML, Image


In [ ]:
# None מחפש אוטומטית. אפשר לציין תיקייה שחולצה, תיקיית האב שלה או ZIP מלא.
# לדוגמה: Path("/content/pin/spno/results/phase6-standalone-artifacts")
SOURCE_ROOT = Path(os.environ["SPNO_SOURCE_ROOT"]) if os.environ.get("SPNO_SOURCE_ROOT") else None
SOURCE_SEARCH_ROOTS = OPTIONS.get("source_search_roots", [
    str(PROJECT_ROOT / "results"), "/content/pin/spno/results", "/content/drive/MyDrive",
])
EXTRACTION_ROOT = Path("/content/spno-phase6-imports") if IN_COLAB else PROJECT_ROOT / "results/phase6-imports"
SOURCE_SEARCH_ROOTS.append(str(EXTRACTION_ROOT))
OUTPUT_ROOT = Path(os.environ.get("SPNO_OUTPUT_ROOT", "/content/drive/MyDrive/spno/workflow" if IN_COLAB else str(PROJECT_ROOT / "results/colab-workflow")))
CACHE_ROOT = Path("/content/spno-data-cache") if IN_COLAB else None
# אם המקור עדיין בדיסק הזמני של הריצה הישנה: שנו את SOURCE_ROOT לנתיבו.
# COPY_TO מעתיק ל-Drive ובודק שלמות; None משאיר את המקור במקומו.
COPY_TO = None
# קובץ אופציונלי עם {"data": {...}, "train": {...}} מהאימון המקורי.
SOURCE_CONFIG = os.environ.get("SPNO_SOURCE_CONFIG")
# אפשר להצהיר כאן על TrainConfig המקורי אם הדוח אינו מכיל את תקציב האימון.
# לדוגמה, רק אם זו אכן הפקודה שרצה: {"epochs": 80, "batch_size": 256, "learning_rate": 1e-3, "patience": 8}
SOURCE_TRAIN_CONFIG = None
SEEDS = OPTIONS.get("seeds")  # None = כל ה-seeds שהתגלו בפייז 6
DEVICE = OPTIONS.get("device", "auto")
ALLOW_BUDGET_BOUND = OPTIONS.get("allow_budget_bound", False)

# שינוי הניסוי החדש בלבד; אינו משנה את הצהרת פרוטוקול המקור.
TRAIN_OVERRIDES = OPTIONS.get("train_overrides", {})


In [ ]:
from dataclasses import asdict, replace
# Refresh this module when an older bundle was already imported in this runtime.
import spno.artifacts as artifact_tools
importlib.reload(artifact_tools)
from spno.artifacts import atomic_json, copy_standalone, resolve_standalone_source
from spno.config import DataConfig
from spno.train import TrainConfig
from spno.workflow import Workflow
from spno.experiments import pick_device
from spno.phase_workflow import evaluate_phase

print("Locating full Phase 6 artifacts...")
SOURCE_ROOT = resolve_standalone_source(
    SOURCE_ROOT, search_roots=SOURCE_SEARCH_ROOTS, extraction_root=EXTRACTION_ROOT,
)
print("Resolved source:", SOURCE_ROOT)
if COPY_TO is not None:
    SOURCE_ROOT = copy_standalone(SOURCE_ROOT, Path(COPY_TO))
source_config = json.loads(Path(SOURCE_CONFIG).read_text()) if SOURCE_CONFIG else {}
if SOURCE_TRAIN_CONFIG is not None:
    source_config["train"] = SOURCE_TRAIN_CONFIG
workflow = Workflow(
    SOURCE_ROOT, OUTPUT_ROOT, cache_root=CACHE_ROOT,
    data_config=DataConfig(**source_config["data"]) if "data" in source_config else None,
    train_config=TrainConfig(**source_config["train"]) if "train" in source_config else None,
)
SEEDS = workflow.seeds if SEEDS is None else SEEDS
DEVICE = pick_device(DEVICE)
if workflow.train_config is not None:
    atomic_json(OUTPUT_ROOT / "source-config.json", {
        "data": asdict(workflow.data_config), "train": asdict(workflow.train_config),
    })
import html
rows = workflow.inventory()
headers = ["model", "seed", "converged", "best epoch", "history", "protocol", "architecture"]
body = []
for row in rows:
    values = [row["name"], row["seed"], row["converged"], row["metadata"]["best_epoch"],
              "available" if row["history"] else "not recorded", row["protocol_source"] or "UNKNOWN",
              row["metadata"]["architecture"]]
    body.append("<tr>" + "".join("<td>" + html.escape(str(v)) + "</td>" for v in values) + "</tr>")
display(HTML("<table><tr>" + "".join("<th>" + h + "</th>" for h in headers) + "</tr>" + "".join(body) + "</table>"))
print("Seeds:", SEEDS, "Device:", DEVICE)
print("Original training protocol:", workflow.train_config or "UNKNOWN — complete SOURCE_TRAIN_CONFIG before preparing matched experiments")

if TRAIN_OVERRIDES and workflow.train_config is None:
    raise ValueError("Declare the original training protocol before choosing overrides")
REQUESTED_TRAIN_CONFIG = replace(workflow.train_config, **TRAIN_OVERRIDES) if TRAIN_OVERRIDES else None


## הניסוי ופרוטוקול ההשוואה

| זרוע | פיזיקה | אימון |
|---|---|---|
| A | ללא אילוץ | data loss |
| A+PDE | soft constraint | data loss + lambda × residual |
| B-loop | hard projection לשימור מסה | data loss, עם projection |
| C1 | מבנה משמר פיזיקה | data loss |
| C1+PDE | מבנה + soft constraint | data loss + lambda × residual |

מציגים את **כל** ערכי lambda שנבחרו עבור A וגם עבור C1. ברירת המחדל היא `[0, 0.01, 0.1, 1, 10]`.
ביקורות lambda=0 נכללות תמיד. בתוך כל משפחה, כל lambda מתחיל מאותו אתחול לפי seed — לא ממשקולות שכבר אומנו.
משתמשים באותם נתונים, seeds ופרוטוקול אימון; בחירת checkpoint נעשית לפי שגיאת הנתונים ב־validation בלבד.
C1 שומר על בחירת kinetic/local של מודל המקור. התאמה מלאה מאפשרת שימוש חוזר ב־A, B-loop ו־C1.
בשלושה seeds ברירת המחדל מכילה 33 ניסויים: 9 ביקורות ו־24 ניסויים עם residual חיובי. ביקורות תואמות אינן מאומנות שוב.

**הסתייגות מדעית:** משוואת Crank–Nicolson הדיסקרטית משמרת מסה כאשר פותרים אותה בדיוק; penalty רך אינו מבטיח זאת.
ב־dt סופי הדינמיקה של CN שונה מזו של C1 split-step ושל פותר הייחוס. lambda גדול עלול להוסיף הטיה.
שימור מסה לבדו אינו הוכחה לדיוק בדינמיקה.

מדווחים שגיאה בצעד אחד, שגיאת rollout, סטיית מסה וסטיית אנרגיה באותו אופק `min(100, steps)`.
ה־rollout והאינווריאנטים נמדדים ב־float64 על CPU. הגרף מציג ממוצע וסטיית תקן בין seeds; seed יחיד אינו אומדן לשונות.

## בדיקת מוכנות

checkpoint חסר עוצר את הבדיקה לפני המדידות. הכינו אותו במחברת האימון עם `TRAIN_PHASES=[7]`,
אותם נתיבים, lambdas ו־`TRAIN_OVERRIDES`; בחרו במפורש את מזהי האימון החסרים.
`ALLOW_BUDGET_BOUND=True` מתיר בדיקה חקרנית מסומנת ואינו הופך checkpoint למתכנס.
מקור שאומן בתקציב חסר נשאר כזה; להגדלת התקציב הגדירו פרוטוקול חדש זהה באימון ובבדיקה.


In [ ]:
LAMBDAS = OPTIONS.get("lambdas", [0.0, 0.01, 0.1, 1.0, 10.0])
jobs = workflow.prepare(7, seeds=SEEDS, lambdas=LAMBDAS,
                        train_config=REQUESTED_TRAIN_CONFIG, generate=False)
readiness = workflow.status(jobs)
for row in readiness:
    print(json.dumps(row, ensure_ascii=False))
unavailable = [row for row in readiness if row["status"] != "ready"]
if unavailable:
    ids = [row["id"] for row in unavailable]
    raise RuntimeError(f"Phase 7 checkpoints are not ready. Inspect the statuses above and prepare these IDs in 00_training.ipynb: {ids}")
print(f"Ready: {len(jobs)} experiments across {len(SEEDS)} seeds; horizon = {min(100, workflow.data_config.steps)}")

result = evaluate_phase(
    workflow, 7, seeds=SEEDS, device=DEVICE,
    allow_budget_bound=ALLOW_BUDGET_BOUND, train_config=REQUESTED_TRAIN_CONFIG,
    lambdas=LAMBDAS,
)


In [ ]:
import csv
print("Saved:", result["output"])
print("Exploratory / budget-bound:", result["exploratory"])
print("Selected rollout horizon:", result["selected_horizon"])
print("Physics-loss caveat:", result["confound"])
if not result["comparison_complete"]:
    print("CONTROLS ONLY: no positive lambda was selected; this is not the five-arm comparison.")
if len(SEEDS) < 2:
    print("Single seed: standard deviation is recorded as 0, but variability has not been estimated.")

comparison_path = Path(result["output"]) / "comparison.csv"
with comparison_path.open() as handle:
    comparison_rows = list(csv.DictReader(handle))
columns = ["arm", "lambda", "seed_count", "parameters", "converged_seeds"]
quantities = ["one_step", "rollout", "mass_drift", "energy_drift"]
headers = columns + [q + " (mean ± SD)" for q in quantities]
body = []
for row in comparison_rows:
    values = [row[key] for key in columns]
    values += [f"{float(row[q + '_mean']):.4e} ± {float(row[q + '_std']):.2e}" for q in quantities]
    body.append("<tr>" + "".join("<td>" + html.escape(str(v)) + "</td>" for v in values) + "</tr>")
display(HTML("<table><tr>" + "".join("<th>" + html.escape(h) + "</th>" for h in headers)
             + "</tr>" + "".join(body) + "</table>"))
print("Comparison table:", comparison_path)
for path in sorted((Path(result["output"]) / "plots").glob("*.png")):
    display(Image(filename=str(path)))


## בדיקות המשך: CN oracle, rollout מיושר פאזה ואינווריאנטים באופק ארוך

התא הבא **אינו מאמן דבר**. הוא קורא את אותם checkpoints ומודד ב־float64 על CPU:

1. **CN oracle** — פתרון מדויק של משוואת Crank–Nicolson שה־residual מעניש עליה. מודדים את שגיאת הצעד האחד שלו מול פותר הייחוס, ולכל זרוע את **המרחק בצעד אחד מצעד ה־CN** (`distance_to_cn`) על אותם קלטים. אם המרחק קטן ככל ש־lambda גדל, ה־loss אכן מושך את המודל לדינמיקה הדיסקרטית של CN ולא רק מרחיק אותו מהנתונים. צעד Strang יחיד מוצג כקו ייחוס נוסף. שימו לב: CN בצורה זו משמר בדיוק גם מסה **וגם אנרגיה** דיסקרטית.
2. **שגיאת rollout מיושרת פאזה** — מסירים פאזה גלובלית אחת לכל דגימה. הפער בין השגיאה הגולמית למיושרת הוא החלק שנובע מסחיפת פאזה בלבד. בצעד 100 השגיאה הגולמית צריכה להתאים לטבלת ההשוואה לכל seed.
3. **אינווריאנטים באופק ארוך** — סטיית מסה ואנרגיה עד `LONG_STEPS` צעדים (ברירת מחדל 5000), מעבר ל־200 הפריימים השמורים, רק עבור `LONG_LAMBDAS` (ברירת מחדל: הביקורות, B-loop ו־lambda=0.01). התחזית של משפט הסימפלקטיות: שגיאת אנרגיה חסומה עבור C1, וגדילה סקולרית בלי מבנה. המגמה מותאמת מעבר ל־`fit_from` צעדים; סטייה מתחת ל־1e-10 מסווגת `roundoff`.

1 ו־2 רצים על כל ה־checkpoints (זול). 3 הוא החלק היקר (כ־10–15 דקות ב־CPU של Colab). הדוח נשמר בתיקייה נפרדת `phase7-probes-…` ואינו משנה את דוח ההשוואה.

In [ ]:
from spno.phase_workflow import probe_phase7
from scripts.run_phase7 import probe_table_html

LONG_LAMBDAS = OPTIONS.get("long_lambdas", [0.0, 0.01])  # arms that get the long rollout
LONG_STEPS = OPTIONS.get("long_steps", 5000)             # invariant-only rollout length
LONG_BATCH = OPTIONS.get("long_batch", 32)               # initial conditions from the test split
probes = probe_phase7(workflow, jobs, long_lambdas=LONG_LAMBDAS, long_steps=LONG_STEPS,
                      long_batch=LONG_BATCH, allow_budget_bound=ALLOW_BUDGET_BOUND)
print("Saved:", probes["output"])
print("Exploratory / budget-bound:", probes["exploratory"])
display(HTML(probe_table_html(probes)))
for path in sorted((Path(probes["output"]) / "plots").glob("*.png")):
    display(Image(filename=str(path)))